# 11주차 실습(단순화) - 작은 데이터로 배우는 CNN 기본 실습

목표: PyTorch로 아주 간단한 컨볼루션 신경망(CNN)을 만들고, 작은 데이터셋으로 학습하여 예측 결과를 확인해 봅니다. 난이도: 초급(최하). 전체 실행 시간: 약 30~60분(환경에 따라 다름).

이 노트북의 목적과 예상 소요 시간을 한눈에 보여 줍니다. 수업 시작 전에 학생들이 읽게 하세요.

## 사전 준비
- Python, PyTorch, torchvision, matplotlib 설치 필요
- 권장 커널: CPU로도 동작하지만 GPU가 있으면 더 빠름
- 설치 예: pip install torch torchvision matplotlib

필요한 소프트웨어와 설치 방법을 안내합니다. 설치 오류가 있으면 이 셀을 먼저 확인하세요.

In [1]:
# 1) 필수 라이브러리 불러오기 (셀을 실행해 설치/에러 확인)
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

ModuleNotFoundError: No module named 'torch'

라이브러리 임포트와 디바이스(CPU/GPU) 확인 코드입니다. 오류 시 패키지 설치를 안내하세요.

## 설명
- 원본 세 개의 노트북에서 다루는 내용(데이터셋 불러오기, 모델 정의, 학습/평가, 시각화)을 하나의 간단한 흐름으로 합쳤습니다.
- 학습 시간 단축을 위해 학습/테스트 데이터를 작게 잘라서 사용합니다(학습 2000개, 테스트 500개).

노트북의 전체 흐름과 서브셋 사용 이유를 간단히 적어 두었습니다. 수업 목표를 학생에게 명확히 전달합니다.

In [ ]:
# 2) 데이터 준비 (FashionMNIST의 작은 서브셋 사용)
transform = transforms.ToTensor()
train_full = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# 학습 샘플을 작게 잘라서 사용 -> 수업용으로 빠르게 실행되게 함
train_idx = list(range(2000))
test_idx = list(range(500))
train = Subset(train_full, train_idx)
test = Subset(test_full, test_idx)

train_loader = DataLoader(train, batch_size=64, shuffle=True)
test_loader = DataLoader(test, batch_size=128)

print('train samples:', len(train), 'test samples:', len(test))

데이터셋을 불러오고 학습/테스트를 빠르게 하기 위해 일부만 사용합니다. 배치 크기와 샘플 수를 확인하세요.

In [ ]:
# 3) 데이터 샘플 시각화 (학생들이 데이터 모양을 이해하도록)
images, labels = next(iter(DataLoader(train, batch_size=9, shuffle=True)))
fig, axes = plt.subplots(3,3,figsize=(6,6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.tight_layout()
plt.show()

실제 이미지와 레이블을 보며 입력 데이터의 형태(1채널, 28x28)와 레이블 의미를 확인합니다.

## 간단한 모델 정의
- 매우 작은 CNN: conv -> relu -> pool -> fc
- 교육용으로 복잡한 요소는 제거했습니다.

모델 구조의 큰 그림(합성곱, 활성화, 풀링, 완전연결)을 한 줄로 요약합니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=0)  # 출력 채널 8 (작게)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 13 * 13, 10)
    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN().to(device)
print(model)

모델 코드를 실행하면 각 층의 출력 형태를 확인하세요. conv 출력채널과 FC 입력 차원에 주의합니다.

In [ ]:
# 4) 학습 루프(매우 단순)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
epochs = 3  # 수업용으로 짧게 설정

for epoch in range(epochs):
    model.train()
    total = 0
    correct = 0
    loss_sum = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        preds = out.argmax(dim=1)
        total += yb.size(0)
        correct += (preds == yb).sum().item()
    print(f'Epoch {epoch+1}/{epochs}  loss={loss_sum/total:.4f}  acc={correct/total:.4f}')

학습 루프의 핵심(순전파-손실-역전파-업데이트)과 출력되는 손실/정확도의 의미를 간단히 적어두었습니다.

In [ ]:
# 5) 간단한 평가
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        preds = out.argmax(dim=1)
        total += yb.size(0)
        correct += (preds == yb).sum().item()
print('Test accuracy:', correct/total)

평가 모드와 torch.no_grad()는 학습시 불필요한 연산(그래디언트 계산)을 끄기 위해 사용됩니다. 테스트 정확도는 전체 성능의 지표입니다.

In [ ]:
# 6) 예측 결과 몇 개 시각화
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)
with torch.no_grad():
    out = model(xb)
preds = out.argmax(dim=1).cpu()
xb = xb.cpu()
fig, axes = plt.subplots(2,5,figsize=(10,4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(xb[i].squeeze(), cmap='gray')
    ax.set_title(f'GT:{yb[i].item()}  Pred:{preds[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

일부 샘플의 정답과 예측을 비교하여 어떤 예제가 틀렸는지 시각적으로 확인합니다. 오답 유형을 토론해 보세요.

## 학생 실습 과제(난이도 최하)
1) 셀 순서대로 실행해서 전체 흐름을 이해하기
2) 학습률(lr)을 0.01 또는 0.5로 바꿔보고 결과(학습 곡선, 정확도) 비교하기
3) conv 출력 채널을 8 -> 16으로 바꿔보고 테스트 정확도 변화 확인하기
4) epochs를 1로 줄이거나 5로 늘려서 실행 시간과 성능 차이 관찰하기
5) (추가) train 서브셋 크기를 2000 -> 5000으로 늘려 실행해보기(시간 증가)

학습 목표: 데이터 로딩, 모델 정의, 학습 루프, 평가, 간단한 하이퍼파라미터 변경을 통해 모델 동작 원리를 체감하는 것.

각 과제는 실험을 통해 모델의 동작을 이해하도록 설계되었습니다. 변경 전후 결과를 기록해 토론하세요.